# Load packages

In [1]:
# Import packages
from neuromaps import transforms, nulls, datasets, plotting, images, resampling, stats, parcellate; # import neuromaps and its submodules
from neuromaps.datasets import fetch_fsaverage, fetch_fslr;
from surfplot import Plot; # this package will be used to plot the data on a brain surface
from nilearn import plotting as nplot; # this package will be used to plot the data on a brain volume
import nibabel as nib;
from nibabel import freesurfer
from nilearn.datasets import fetch_atlas_surf_destrieux, load_fsaverage, load_fsaverage_data
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np
from IPython.display import HTML
import os
os.environ["PATH"] += ":/Users/guofanhua/Desktop/gfh/tools/workbench/bin_macosx64"
import pandas as pd


# Data load and preprocess

## load surface data 

In [2]:
# Fetch surface data of fsaverage we wil use for plotting with 10k vertices
surfaces_fsaverage_10 = fetch_fsaverage(density='10k') 
lh_fsaverage_10, rh_fsaverage_10 = surfaces_fsaverage_10['inflated'] # there are many types of surfaces, we will use inflated surface for easy visualization of gyri and sulci

# # Fetch surface data of fsaverage we wil use for plotting with 41k vertices
# surfaces_fsaverage_41 = fetch_fsaverage(density='41k') 
# lh_fsaverage_41, rh_fsaverage_41 = surfaces_fsaverage_41['inflated'] # there are many types of surfaces, we will use inflated surface for easy visualization of gyri and sulci

## load parcellation data

In [3]:
# Get HCP-MMP1.0 atlas
def prep_hcp_mmp_parcellation(annot_left_path , annot_right_path):
    '''Takes the path to the right and left hemisphere annotation files
    and loads everything.
    '''
    hcp_mmp = {}
    annot_left = freesurfer.read_annot(annot_left_path)
    annot_right = freesurfer.read_annot(annot_right_path)
    labels=annot_left[2]
    map_left=annot_left[0]
    map_right=annot_right[0]

    hcp_mmp['labels'] = labels
    
    hcp_mmp['map_left'] = map_left
    hcp_mmp['map_right'] = map_right

    print(f"Keys available: {sorted(hcp_mmp)}")
    print(f"Number of vertices in each hemisphere: {len(hcp_mmp['map_left'])} , {len(hcp_mmp['map_right'])}")
    print(f"Number of labels: {len(hcp_mmp['labels'])}")

    return hcp_mmp

# sample use
annot_left_path = "/Users/guofanhua/Desktop/gfh/work/StandardBrainTemplateAndAtlas/AllenBrain/microarray/atlas/fsa5_lh_glasser_360.annot"
annot_right_path = "/Users/guofanhua/Desktop/gfh/work/StandardBrainTemplateAndAtlas/AllenBrain/microarray/atlas/fsa5_rh_glasser_360.annot"
hcp_mmp = prep_hcp_mmp_parcellation(annot_left_path , annot_right_path)

labels = [label.decode() for label in hcp_mmp['labels']]
parc_left = images.construct_shape_gii(hcp_mmp['map_left'], labels=labels, intent='NIFTI_INTENT_LABEL')
parc_right = images.construct_shape_gii(hcp_mmp['map_right'], labels=labels, intent='NIFTI_INTENT_LABEL')
parcellation = images.relabel_gifti((parc_left, parc_right), background=['medialwall'])
parcellation = transforms.fsaverage_to_fsaverage(parcellation, target_density='10k', method='nearest')
print(parcellation)
hcp_mmp_parc = parcellate.Parcellater(parcellation=parcellation, space='fsaverage').fit() # create the parcellator for the HCP_MMP atlas

# # plot the hcp_mmp1_0 atlas resampled on the fsaverage brain
# p = Plot(surf_lh=lh_fsaverage_10, surf_rh=rh_fsaverage_10)
# p.add_layer(images.load_data(parcellation), cmap='viridis')
# fig = p.build()
# fig.axes[0].set_title('HCP-MMP', pad=-3)
# fig.show()

Keys available: ['labels', 'map_left', 'map_right']
Number of vertices in each hemisphere: 10242 , 10242
Number of labels: 181
(<nibabel.gifti.gifti.GiftiImage object at 0x7fb88983dcd0>, <nibabel.gifti.gifti.GiftiImage object at 0x7fb88983df70>)


## load some other data 

In [10]:
# get all publish other data
def data_to_parcellation(annotation, fname , hcp_mmp_parc):
    if fname[2]=='MNI152':
        data_fsaverage = transforms.mni152_to_fsaverage(annotation, fsavg_density='10k', method='linear')
    if fname[2]=='fsaverage':
        data_fsaverage = transforms.fsaverage_to_fsaverage(annotation, target_density='10k', method='linear')
    if fname[2]=='fsLR':
        data_fsaverage = transforms.fslr_to_fsaverage(annotation, target_density='10k', method='linear')
    if fname[2]=='civet':
        data_fsaverage = transforms.civet_to_fsaverage(annotation, target_density='10k', method='linear')
    data_parc = hcp_mmp_parc.transform(data_fsaverage, space='fsaverage')
    return data_parc


all_datasets = datasets.available_annotations(space='all')  
all_other_data = []
for fname in all_datasets:
    source, desc, space, den = fname  
    try:
        tmp_annotation = datasets.fetch_annotation(source=source, desc=desc, space=space, den=den, verbose=0)
        data_parc = data_to_parcellation(tmp_annotation, fname, hcp_mmp_parc)
        all_other_data.append((source, desc, space, den, data_parc))
    except Exception as e:
        print(f"❌ Failed fetching {fname}: {e}")
        continue



(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
❌ Failed fetching ('hill2010', 'devexp', 'fsLR', '164k'): Must specify `hemi` when only 1 data file supplied
❌ Failed fetching ('hill2010', 'evoexp', 'fsLR', '164k'): Must specify `hemi` when only 1 data file supplied
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)
(360,)


In [ ]:
df = pd.DataFrame([
    [source, desc, space, den] + list(data_parc)
    for (source, desc, space, den, data_parc) in all_other_data])

# 设置列名
df.columns = ['source', 'desc', 'space', 'den'] + [f'parcel_{i}' for i in range(360)]

# 保存
df.to_csv('/Users/guofanhua/Desktop/gfh/work/experiment/ASL_Mesoscopic2025/reference/all_other_data.csv', index=False)